In [ ]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '.')

from Engine.TickData import TickData
from Engine.Backtester import Backtester

In [ ]:
# Load ES tick data and aggregate into 250-tick bars
td = TickData('./Data/ESTick.txt', timezone='America/Chicago')
td.aggregate(method='tick', size=250)

In [ ]:
# ES contract specs
tick_value = 12.50        # $12.50 per tick (0.25 pts)
contract_multiplier = 50  # $50 per point
initial_equity = 100_000

ohlc = td.get_ohlc()
indices = td.get_indices()
timeframe = td.get_timeframe()
n_bars = len(ohlc)

print(f'Bars: {n_bars:,}')
print(f'Timeframe: {timeframe}')
print(f'Date range: {indices[0]} to {indices[-1]}')

In [ ]:
# Coin flip strategy:
# - Every 100 bars, if flat, flip a coin to go long (+1) or short (-1)
# - TP = 10 ticks, SL = 10 ticks (1:1 RR)
# - ES: 1 tick = 0.25 pts, so 10 ticks = 2.5 points

tp_ticks = 10
sl_ticks = 10
tp_points = tp_ticks * 0.25  # 2.5 points
sl_points = sl_ticks * 0.25  # 2.5 points

np.random.seed(42)

signal = np.zeros(n_bars)
position = 0
entry_price = 0.0
bars_since_last_trade = 0

for i in range(n_bars):
    close = ohlc[i, 3]
    high = ohlc[i, 1]
    low = ohlc[i, 2]

    if position != 0:
        hit_tp = False
        hit_sl = False

        if position == 1:
            hit_tp = high >= entry_price + tp_points
            hit_sl = low <= entry_price - sl_points
        else:
            hit_tp = low <= entry_price - tp_points
            hit_sl = high >= entry_price + sl_points

        if hit_tp or hit_sl:
            position = 0
            bars_since_last_trade = 0
    else:
        bars_since_last_trade += 1
        if bars_since_last_trade >= 100:
            position = np.random.choice([1, -1])
            entry_price = close
            bars_since_last_trade = 0

    signal[i] = position

print(f'Total entries: {int(np.sum(np.abs(np.diff(signal)) > 0))}')
print(f'Long entries: {int(np.sum(np.diff(signal) > 0))}')
print(f'Short entries: {int(np.sum(np.diff(signal) < 0))}')

In [ ]:
# Run backtester — no slippage, $5 round trip commission
bt = Backtester(
    ohlc=ohlc,
    indicies=indices,
    signal=signal,
    timeframe=timeframe,
    tick_value=tick_value,
    contract_multiplier=contract_multiplier,
    initial_equity=initial_equity,
    slippage_ticks=0,
    commission=5.0,
)
bt.full_backtest()
print(bt)

In [ ]:
# Equity curve plot
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(indices, bt.equity_curve, linewidth=0.8, color='blue')
ax1.axhline(initial_equity, color='gray', linestyle='--', alpha=0.5)
ax1.set_ylabel('Equity ($)')
ax1.set_title('Coin Flip Strategy - ES 250 Tick | 10 tick TP/SL (1:1) | $5 commission')
ax1.grid(True, alpha=0.3)

ax2.fill_between(indices, bt.drawdown_curve, 0, color='red', alpha=0.4)
ax2.set_ylabel('Drawdown ($)')
ax2.set_xlabel('Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate interactive HTML report
from Engine.Report import generate_report

generate_report(bt, 'backtest_report.html', open_browser=True)